<a href="https://colab.research.google.com/github/dordor7/SkillReal-Home-Assignment/blob/main/SkillReal_ML_Engineer_Home_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Install rdkit (often required for chemical informatics tools like LoHi)
!pip install rdkit

In [20]:
import os
import subprocess

# Clone the repository if it doesn't already exist
repo_dir = "/content/LoHi-Weld"
if not os.path.exists(repo_dir):
    print("Cloning LoHi-Weld repository...")
    subprocess.run(["git", "clone", "https://github.com/SylvioBlock/LoHi-Weld.git", repo_dir], check=True)
else:
    print("Repository already cloned.")

# Change working directory to the project folder
os.chdir(repo_dir)
print(f"Current working directory: {os.getcwd()}")

Repository already cloned.
Current working directory: /content/LoHi-Weld


In [21]:
# Clean requirements.txt of absolute/local conda paths AND remove strict version pinning to prevent compilation issues
import re

with open('requirements.txt', 'r') as f:
    lines = f.readlines()

cleaned_lines = []
for line in lines:
    line_str = line.strip()
    # Skip empty lines or comments
    if not line_str or line_str.startswith('#'):
        continue
    # Skip absolute local paths
    if line_str.startswith('/') or 'file://' in line_str:
        continue

    # Strip out strict '==' version pins so pip fetches modern, pre-compiled wheels compatible with python 3.13
    package_name = re.split(r'==|>=|<=|>|<', line_str)[0].strip()
    if package_name:
        cleaned_lines.append(package_name + '\n')

with open('requirements_cleaned.txt', 'w') as f:
    f.writelines(cleaned_lines)

!pip install -r requirements_cleaned.txt

In [22]:
import gdown
import zipfile

# Google Drive file ID from the provided URL
file_id = '1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK'
download_url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'weld-dataset.zip'

print("Downloading the dataset from Google Drive...")
gdown.download(download_url, output_zip, quiet=False)

print("Extracting the dataset...")
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall('.')

# Optional: Clean up the downloaded ZIP file
if os.path.exists(output_zip):
    os.remove(output_zip)

print("Dataset download and extraction complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK
From (redirected): https://drive.google.com/uc?id=1pXeEnREfV_MYcL5MY2vkd9njBm_blPUK&confirm=t&uuid=b2e84ae0-92eb-4f64-abfb-570df48fad3e
To: /content/LoHi-Weld/weld-dataset.zip
100%|██████████| 631M/631M [00:13<00:00, 45.1MB/s]


Extracting the dataset...
Dataset download and extraction complete!


In [23]:
# Remove annotations for classes other than 'pore' and 'deposit'

import json
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")

for subset in ("low_resolution_welds", "high_resolution_welds"):
    folder = root / subset

    for path in folder.glob("*.yolo"):
        lines = path.read_text(encoding="utf-8").splitlines()
        kept = [
            line for line in lines
            if line.strip() and line.split()[0] in {"0", "1"}
        ]
        path.write_text("".join(line + "\n" for line in kept), encoding="utf-8")

    for path in folder.glob("*.json"):
        annotation = json.loads(path.read_text(encoding="utf-8"))
        annotation["shapes"] = [
            shape for shape in annotation["shapes"]
            if shape["label"] in {"pore", "deposit"}
        ]
        path.write_text(json.dumps(annotation, indent=2) + "\n", encoding="utf-8")

    print(f"Filtered {subset}")

Filtered low_resolution_welds
Filtered high_resolution_welds


In [24]:
# Delete default kfold splits
import shutil
from pathlib import Path

root = Path("/content/LoHi-Weld/weld-dataset")
targets = [root / "kfold_high", root / "kfold_low"]

for path in targets:
    if path.exists() and path.is_dir():
        shutil.rmtree(path)
        print(f"Successfully deleted: {path}")
    elif path.exists() and not path.is_dir():
        path.unlink()
        print(f"Successfully deleted file: {path}")
    else:
        print(f"Path does not exist (already deleted): {path}")

Successfully deleted: /content/LoHi-Weld/weld-dataset/kfold_high
Successfully deleted: /content/LoHi-Weld/weld-dataset/kfold_low


In [25]:
# Import new kfold split scripts
import gdown
import os

# Google Drive IDs for the files
py_file_id = '18dFQmRzoSuVhghfYyPxxPMsP2XH5vEz7'
sh_file_id = '1EueWa9l9JW64BqB7I_Vqy1PU68Nki448'

py_download_url = f'https://drive.google.com/uc?id={py_file_id}'
sh_download_url = f'https://drive.google.com/uc?id={sh_file_id}'

target_py = '/content/LoHi-Weld/weld-dataset/create_kfold.py'
target_sh = '/content/LoHi-Weld/weld-dataset/create_kfold.sh'

print("Downloading and replacing create_kfold.py inside weld-dataset...")
gdown.download(py_download_url, target_py, quiet=False)

print("Downloading and replacing create_kfold.sh inside weld-dataset...")
gdown.download(sh_download_url, target_sh, quiet=False)

# Make the shell script executable
if os.path.exists(target_sh):
    os.chmod(target_sh, 0o755)
    print(f"Permissions set: {target_sh} is now executable.")

print("Files successfully placed in weld-dataset!")

Downloading...
From (original): https://drive.google.com/uc?id=18dFQmRzoSuVhghfYyPxxPMsP2XH5vEz7
From (redirected): https://drive.google.com/uc?id=18dFQmRzoSuVhghfYyPxxPMsP2XH5vEz7&confirm=t&uuid=a14316ef-83c3-490d-8357-4b66e72b1fcb
To: /content/LoHi-Weld/weld-dataset/create_kfold.py
100%|██████████| 4.85k/4.85k [00:00<00:00, 8.14MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1EueWa9l9JW64BqB7I_Vqy1PU68Nki448
From (redirected): https://drive.google.com/uc?id=1EueWa9l9JW64BqB7I_Vqy1PU68Nki448&confirm=t&uuid=4edadca0-a2b0-4544-a715-b54956e902e1
To: /content/LoHi-Weld/weld-dataset/create_kfold.sh
100%|██████████| 2.47k/2.47k [00:00<00:00, 5.06MB/s]

Permissions set: /content/LoHi-Weld/weld-dataset/create_kfold.sh is now executable.
Files successfully placed in weld-dataset!


In [26]:
# Create new kfolds
!OVERSAMPLE=0 bash weld-dataset/create_kfold.sh both

Dataset folder: /content/LoHi-Weld/weld-dataset
Python script: /content/LoHi-Weld/weld-dataset/create_kfold.py

Building kfold_high
Source: 1022 JPGs, 194 without pores or deposits
kfold_high/test: 205 images (0 repeats), 39 without pores or deposits, pores: 46 images / 98 boxes, deposits: 157 images / 376 boxes
fold0/train: 653 images (0 repeats), 124 without pores or deposits, pores: 146 images / 349 boxes, deposits: 499 images / 1160 boxes
fold0/val: 164 images (0 repeats), 31 without pores or deposits, pores: 37 images / 76 boxes, deposits: 125 images / 258 boxes
fold1/train: 653 images (0 repeats), 124 without pores or deposits, pores: 146 images / 341 boxes, deposits: 499 images / 1126 boxes
fold1/val: 164 images (0 repeats), 31 without pores or deposits, pores: 37 images / 84 boxes, deposits: 125 images / 292 boxes
fold2/train: 654 images (0 repeats), 124 without pores or deposits, pores: 147 images / 347 boxes, deposits: 499 images / 1127 boxes
fold2/val: 163 images (0 repeats)

In [27]:
# Change yamls at /data to 2 classes - 'pore', 'deposit'
import yaml
from pathlib import Path

data_dir = Path("/content/LoHi-Weld/data")
yaml_files = list(data_dir.glob("*.yaml")) + list(data_dir.glob("*.yml"))

# Filter out hyperparameter configuration files
yaml_files = [f for f in yaml_files if "hyp" not in f.name]

print(f"Found {len(yaml_files)} YAML config files to update in {data_dir} (excluding hyp files).")

for yaml_file in yaml_files:
    try:
        # Load the configuration
        with open(yaml_file, 'r', encoding='utf-8') as f:
            config = yaml.safe_load(f) or {}

        # Update class attributes
        config['nc'] = 2
        config['names'] = ['pore', 'deposit']

        # Save the updated configuration back
        with open(yaml_file, 'w', encoding='utf-8') as f:
            yaml.safe_dump(config, f, default_flow_style=False, sort_keys=False)

        print(f"Successfully updated: {yaml_file.name}")
    except Exception as e:
        print(f"Error processing {yaml_file.name}: {e}")

Found 10 YAML config files to update in /content/LoHi-Weld/data (excluding hyp files).
Successfully updated: low.fold3.yaml
Successfully updated: low.fold4.yaml
Successfully updated: high.fold1.yaml
Successfully updated: low.fold0.yaml
Successfully updated: high.fold3.yaml
Successfully updated: low.fold1.yaml
Successfully updated: high.fold0.yaml
Successfully updated: low.fold2.yaml
Successfully updated: high.fold4.yaml
Successfully updated: high.fold2.yaml


In [28]:
# Debugging training errors
import subprocess
from pathlib import Path

google_utils_path = Path("/content/LoHi-Weld/utils/google_utils.py")

# 1. Revert to clean original state
subprocess.run(["git", "checkout", "--", str(google_utils_path)], check=True)
content = google_utils_path.read_text(encoding='utf-8')

# 2. Define a clean, single-line inline safe patch that uses try/except expression style or lambda
old_line = "tag = subprocess.check_output('git tag', shell=True).decode().split()[-1]"
new_line = "tag = (subprocess.check_output('git tag', shell=True).decode().split() or ['v0.1'])[-1]"

if old_line in content:
    content = content.replace(old_line, new_line)
    google_utils_path.write_text(content, encoding='utf-8')
    print("Successfully patched google_utils.py with a robust single-line fallback!")
else:
    print("Target line not found in google_utils.py!")

from pathlib import Path

metrics_path = Path("/content/LoHi-Weld/utils/metrics.py")
if metrics_path.exists():
    content = metrics_path.read_text(encoding='utf-8')
    if "np.trapz" in content:
        content = content.replace("np.trapz", "np.trapezoid")
        metrics_path.write_text(content, encoding='utf-8')
        print("Successfully replaced np.trapz with np.trapezoid in metrics.py!")
    else:
        print("np.trapz not found in metrics.py (already replaced?)")
else:
    print(f"Could not locate {metrics_path}")

Successfully patched google_utils.py with a robust single-line fallback!
Successfully replaced np.trapz with np.trapezoid in metrics.py!


In [29]:
# Check no. of CPU cores
import os
print(len(os.sched_getaffinity(0)))

12


In [30]:
# Clean previous runs' data
!./clean.sh

In [31]:
# Training smoke test
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!python -u train.py \
    --data data/high.fold0.yaml \
    --cfg cfg/training/yolov7-tiny.yaml \
    --weights yolov7-tiny.pt \
    --hyp data/hyp.data.aug.tiny.yaml \
    --img 640 640 \
    --batch-size 128 \
    --workers 8 \
    --device 0 \
    --epochs 1 \
    --name high.fold0.aug.640.smoke

I0000 00:00:1794559936.018203   35171 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794559936.087078   35171 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794559938.316282   35171 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794559938.316862   35171 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [32]:
# Train run
import os

os.environ["WANDB_MODE"] = "disabled"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

for fold in range(5):
    print(f"\n=== Training low.fold{fold} ===", flush=True)

    !python -u train.py \
        --data data/low.fold{fold}.yaml \
        --cfg cfg/training/yolov7-tiny.yaml \
        --weights yolov7-tiny.pt \
        --hyp data/hyp.data.aug.tiny.yaml \
        --img 640 640 \
        --batch-size 128 \
        --workers 8 \
        --device 0 \
        --epochs 50 \
        --name low.fold{fold}.aug.640.e50

    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Training failed on fold {fold}")


=== Training low.fold0 ===
I0000 00:00:1794560027.275560   35691 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794560027.347442   35691 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794560029.591462   35691 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794560029.591995   35691 cudart_stub.cc:31] Could not find cuda drivers on your ma

In [33]:
# Test run
%cd /content/LoHi-Weld

import os
from pathlib import Path

os.environ["WANDB_MODE"] = "disabled"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

checkpoints = {
    fold: Path(f"runs/train/low.fold{fold}.aug.640.e50/weights/best.pt")
    for fold in range(5)
}

missing = [str(path) for path in checkpoints.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Check the training run folder names:\n" + "\n".join(missing))

for fold, weights in checkpoints.items():
    print(f"\n=== Testing low-res fold {fold} ===", flush=True)
    !python -u test.py \
        --task test \
        --data data/low.fold{fold}.yaml \
        --weights {weights} \
        --img-size 640 \
        --batch-size 128 \
        --device 0 \
        --conf-thres 0.001 \
        --iou-thres 0.50 \
        --verbose \
        --save-txt \
        --save-conf \
        --name low.fold{fold}.aug.640.e50.holdout

    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Testing failed on fold {fold}")

/content

=== Testing low-res fold 0 ===
Namespace(weights=['runs/train/low.fold0.aug.640.e50/weights/best.pt'], data='data/low.fold0.yaml', batch_size=128, img_size=640, conf_thres=0.001, iou_thres=0.5, task='test', device='0', single_cls=False, augment=False, verbose=True, save_txt=True, save_hybrid=False, save_conf=True, save_json=False, project='runs/test', name='low.fold0.aug.640.e50.holdout', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 14e7b7a torch 2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563.125MB)

/content/LoHi-Weld/models/experimental.py:252: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  ckpt = torch.load(w, map_location=map_location)  # load
Fusing layers... 
IDetect.fuse
Model Summary: 208 layers, 6010302 parameters, 0 gradients
 Convert model to Traced-model... 
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: Futu

In [34]:
# Saving the run results
import shutil
from google.colab import files

# Compress the runs folder
runs_zip = "runs_backup.zip"
print("Compressing the runs folder, please wait...")
shutil.make_archive(runs_zip.replace(".zip", ""), 'zip', "runs")

# Trigger browser download of the zip file
print("Starting download...")
files.download(runs_zip)

Compressing the runs folder, please wait...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
# Importing the result from my previous run
import gdown
import os

# Google Drive RAR file link: https://drive.google.com/file/d/1dprhQyf-fBUI1-R21FziDQB9H6ITCnQK/view?usp=sharing
file_id = '1dprhQyf-fBUI1-R21FziDQB9H6ITCnQK'
download_url = f'https://drive.google.com/uc?id={file_id}'
output_rar = '/content/LoHi-Weld/runs_backup.rar'
target_dir = '/content/LoHi-Weld/runs'

# Ensure target directory exists
os.makedirs(target_dir, exist_ok=True)

print("Downloading the pre-trained runs RAR archive from Google Drive...")
gdown.download(download_url, output_rar, quiet=False)

# Extracting RAR using system-level unrar command into target directory
print("Extracting the runs archive into LoHi-Weld/runs...")
!unrar x -o+ "{output_rar}" "/content/LoHi-Weld/"

# Clean up the downloaded RAR file
if os.path.exists(output_rar):
    os.remove(output_rar)

print(f"Extraction complete! Runs folder ready.")

Downloading...
From (original): https://drive.google.com/uc?id=1dprhQyf-fBUI1-R21FziDQB9H6ITCnQK
From (redirected): https://drive.google.com/uc?id=1dprhQyf-fBUI1-R21FziDQB9H6ITCnQK&confirm=t&uuid=49671fc0-90d4-4138-9a24-89c52493ca47
To: /content/LoHi-Weld/runs_backup.rar
100%|██████████| 1.52G/1.52G [00:37<00:00, 40.9MB/s]


Extracting the runs archive into LoHi-Weld/runs...

UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from /content/LoHi-Weld/runs_backup.rar

Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/confusion_matrix.png       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/F1_curve.png       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/labels/200905_165109_0000000001_CAM1_NG_1.txt       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/labels/200907_064553_0000000001_CAM1_NG_0.txt       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/labels/200907_065005_0000000001_CAM1_NG_1.txt       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/labels/200907_071105_0000000002_CAM1_NG_0.txt       0%  OK 
Extracting  /content/LoHi-Weld/runs/test/low.f

In [36]:
"""Summarize LoHi-Weld YOLOv7 detections for horizontal weld-bead images."""

import argparse
import json
from pathlib import Path
from statistics import mean, median

from PIL import Image


CLASS_NAMES = {0: "pore", 1: "deposit"}


def read_predictions(label_path, width, height, min_conf):
    """Read YOLOv7 --save-txt --save-conf rows and convert boxes to image pixels."""
    if not label_path.exists():
        return

    for line_number, line in enumerate(label_path.read_text().splitlines(), 1):
        if not line.strip():
            continue
        try:
            class_id, cx, cy, box_width, box_height, score = map(float, line.split())
            if class_id not in CLASS_NAMES:
                raise ValueError("expected class 0 (pore) or 1 (deposit)")
            if not all(0 <= value <= 1 for value in (cx, cy, box_width, box_height, score)):
                raise ValueError("expected normalized coordinates and confidence in [0, 1]")
        except ValueError as exc:
            raise ValueError(f"{label_path}:{line_number}: {exc}") from exc

        if score < min_conf or box_width == 0 or box_height == 0:
            continue

        left = max(0, (cx - box_width / 2) * width)
        right = min(width, (cx + box_width / 2) * width)
        top = max(0, (cy - box_height / 2) * height)
        bottom = min(height, (cy + box_height / 2) * height)
        yield CLASS_NAMES[class_id], [left, top, right, bottom], score


def densest_window(positions, left, right, length):
    """Find the maximum count of centers in a closed interval of fixed length."""
    positions = sorted(positions)
    if not positions:
        return {"count": 0, "x_range_px": None}

    start = 0
    best_count = 0
    best_start = left
    for end, x in enumerate(positions):
        while positions[start] < x - length - 1e-9:
            start += 1
        count = end - start + 1
        if count > best_count:
            best_count = count
            best_start = min(positions[start], right - length)
    return {"count": best_count, "x_range_px": [best_start, best_start + length]}


def summarize_image(image_path, labels_dir, min_conf, section_fraction, bead_boxes):
    with Image.open(image_path) as image:
        width, height = image.size

    left, top, right, bottom = bead_boxes.get(image_path.stem, [0, 0, width, height])
    if not (0 <= left < right <= width and 0 <= top < bottom <= height):
        raise ValueError(f"Invalid bead box for {image_path.name}")

    centerline_y = (top + bottom) / 2
    section_length = (right - left) * section_fraction
    label_path = labels_dir / f"{image_path.stem}.txt"

    pores = []
    deposits = []
    for defect, box, score in read_predictions(label_path, width, height, min_conf):
        x1, y1, x2, y2 = box
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        if not (left <= cx <= right and top <= cy <= bottom):
            continue

        record = {
            "box_xyxy_px": box,
            "confidence": score,
            "area_px2": (x2 - x1) * (y2 - y1),
        }
        if defect == "pore":
            pores.append(record)
        else:
            record["distance_to_centerline_px"] = max(
                y1 - centerline_y, centerline_y - y2, 0
            )
            deposits.append(record)

    areas = [pore["area_px2"] for pore in pores]
    distances = [deposit["distance_to_centerline_px"] for deposit in deposits]
    positions = [(pore["box_xyxy_px"][0] + pore["box_xyxy_px"][2]) / 2 for pore in pores]

    # When distances tie, use the deposit with the largest bounding box.
    closest_deposit = min(
        deposits,
        key=lambda deposit: (
            deposit["distance_to_centerline_px"],
            -deposit["area_px2"],
        ),
        default=None,
    )

    return {
        "image": image_path.name,
        "image_size_px": [width, height],
        "bead_box_xyxy_px": [left, top, right, bottom],
        "bead_box_source": "provided_box" if image_path.stem in bead_boxes else "image_crop",
        "centerline_y_px": centerline_y,
        "pore_count": len(pores),
        "deposit_count": len(deposits),
        "closest_deposit_area_px2": (
            closest_deposit["area_px2"] if closest_deposit is not None else None
        ),
        "pore_box_area_px2": {
            "max": max(areas, default=None),
            "mean": mean(areas) if areas else None,
            "median": median(areas) if areas else None,
        },
        "pore_density": {
            "section_length_px": section_length,
            "densest_window": densest_window(positions, left, right, section_length),
        },
        "deposit_distance_to_centerline_px": {
            "min": min(distances, default=None),
            "mean": mean(distances) if distances else None,
            "median": median(distances) if distances else None,
        },
        "pores": pores,
        "deposits": deposits,
    }


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--images", type=Path, required=True, help="Directory of JPG input images")
    parser.add_argument("--labels", type=Path, required=True, help="YOLOv7 prediction labels directory")
    parser.add_argument("--output", type=Path, required=True, help="Output JSON file")
    parser.add_argument("--min-conf", type=float, required=True, help="Minimum confidence in [0, 1]")
    parser.add_argument(
        "--section-fraction", type=float, required=True,
        help="Window length as a fraction of the bead box's x-span, in (0, 1]",
    )
    parser.add_argument(
        "--bead-boxes", type=Path,
        help="Optional JSON mapping image stems to bead boxes [x1, y1, x2, y2] in pixels",
    )
    args = parser.parse_args()

    if not 0 <= args.min_conf <= 1:
        parser.error("--min-conf must be in [0, 1]")
    if not 0 < args.section_fraction <= 1:
        parser.error("--section-fraction must be in (0, 1]")
    if not args.images.is_dir() or not args.labels.is_dir():
        parser.error("--images and --labels must be existing directories")

    images = sorted(
        path for path in args.images.iterdir()
        if path.is_file() and path.suffix.lower() == ".jpg"
    )
    if not images:
        parser.error(f"No JPG images found in {args.images}")
    image_stems = {image.stem for image in images}
    if len(image_stems) != len(images):
        parser.error("Image filenames must have unique stems to match YOLO label files")

    label_stems = {path.stem for path in args.labels.glob("*.txt")}
    if label_stems - image_stems:
        parser.error("Some prediction files do not match the supplied images")
    if not label_stems:
        print("No prediction files found. Verify that this is the completed prediction run.")

    bead_boxes = json.loads(args.bead_boxes.read_text()) if args.bead_boxes else {}
    reports = [
        summarize_image(image, args.labels, args.min_conf, args.section_fraction, bead_boxes)
        for image in images
    ]
    output = {
        "min_confidence": args.min_conf,
        "section_fraction": args.section_fraction,
        "images_with_prediction_files": len(label_stems),
        "welds": reports,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(output, indent=2, allow_nan=False) + "\n")
    print(f"Wrote {len(reports)} weld reports to {args.output}")


# Run the CLI when executed as a script. A Colab cell only defines the functions.
if __name__ == "__main__" and "__file__" in globals():
    main()

In [37]:
import json
from pathlib import Path

repo = Path("/content/LoHi-Weld")
images = sorted((repo / "weld-dataset/kfold_low/test/images").glob("*.jpg"))

min_conf = 0.25
section_fraction = 0.20

for fold in range(5):
    run_dir = repo / "runs/test" / f"low.fold{fold}.aug.640.e50.holdout"
    labels_dir = run_dir / "labels"
    if not labels_dir.is_dir():
        raise FileNotFoundError(f"Prediction folder missing: {labels_dir}")

    report = {
        "min_confidence": min_conf,
        "section_fraction": section_fraction,
        "images_with_prediction_files": len(list(labels_dir.glob("*.txt"))),
        "welds": [
            summarize_image(
                image_path=image,
                labels_dir=labels_dir,
                min_conf=min_conf,
                section_fraction=section_fraction,
                bead_boxes={},
            )
            for image in images
        ],
    }

    output = run_dir / "weld_report.json"
    output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
    print(f"Fold {fold}: wrote {len(report['welds'])} welds to {output}")

Fold 0: wrote 400 welds to /content/LoHi-Weld/runs/test/low.fold0.aug.640.e50.holdout/weld_report.json
Fold 1: wrote 400 welds to /content/LoHi-Weld/runs/test/low.fold1.aug.640.e50.holdout/weld_report.json
Fold 2: wrote 400 welds to /content/LoHi-Weld/runs/test/low.fold2.aug.640.e50.holdout/weld_report.json
Fold 3: wrote 400 welds to /content/LoHi-Weld/runs/test/low.fold3.aug.640.e50.holdout/weld_report.json
Fold 4: wrote 400 welds to /content/LoHi-Weld/runs/test/low.fold4.aug.640.e50.holdout/weld_report.json


In [38]:
# Download the dataset and data yamls
import shutil
from google.colab import files

# Compress and download the data and weld-dataset folders
for folder_name in ['data', 'weld-dataset']:
    shutil.make_archive(f'/content/{folder_name}_backup', 'zip', f'/content/LoHi-Weld/{folder_name}')
    print(f'Download started for {folder_name}_backup.zip...')
    files.download(f'/content/{folder_name}_backup.zip')

Download started for data_backup.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started for weld-dataset_backup.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>